In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import solve
from numpy.polynomial.legendre import leggauss
import torch
from scipy.linalg import eigh
from scipy.linalg import solve



In [ ]:
#@title FEM in 1D

def mesh(M,OMEGA):
  l_boundary, r_boundary = OMEGA
  triangles = np.linspace(l_boundary, r_boundary, M+1)
  return triangles

def evaluate_function(triangles, function):
    inner_triangles = triangles[1:-1]
    function_values = np.array([])
    for i in range(len(triangles)):
        function_value = function(triangles[i])
        function_values.append(function_value)
    return function_values

def matrix(M, OMEGA, inner_triangles):
  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M
  stiff_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))
  mass_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))

  for i in range(stiff_matrix.shape[0]):
    stiff_matrix[i,i] = 2
    mass_matrix[i,i] = 4

  for i in range(stiff_matrix.shape[0]-1):
    stiff_matrix[i,i+1] = -1
    stiff_matrix[i+1,i] = -1
    mass_matrix[i,i+1] = 1
    mass_matrix[i+1,i] = 1

  stiff = 1/h * stiff_matrix
  mass = h/6 * mass_matrix

  return stiff, mass

def mass(M, OMEGA, inner_triangles):
  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M
  mass_matrix  = np.zeros((len(inner_triangles),len(inner_triangles)))

  for i in range(mass_matrix.shape[0]):
    mass_matrix[i,i] = 4

  for i in range(mass_matrix.shape[0]-1):
    mass_matrix[i,i+1] = 1
    mass_matrix[i+1,i] = 1

  mass = h/6 * mass_matrix

  return mass


def calculate_RHS(f, func, M, OMEGA, t):

  l_boundary, r_boundary = OMEGA
  h = (r_boundary - l_boundary)/M

  tri = func(M, OMEGA)
  RHS = np.zeros(M+1)

  for i in range(M):
    Integral_fphik = (1/2) * (tri[i+1] - tri[i]) * f(tri[i] + (tri[i+1] - tri[i])*(1/2), t)  #Q(f) = abs(T)*f(x_s)
    RHS[i] = RHS[i] + Integral_fphik
    RHS[i+1] = RHS[i+1] + Integral_fphik
  return RHS[1:-1]


In [ ]:
#@title discrete operator for Au =-$\nabla\cdot$(a(x) $\nabla$u)

# stiffnessmatrix for div(a grad(u))

def dsicrete_operator_div_a_grad_u(M, OMEGA, inner_triangles, a, quad_order = 3):

  l_boundary, r_boundary = OMEGA

  xi_q, w_q = leggauss(quad_order)

  tri = mesh(M, OMEGA)

  stiff = np.zeros((M+1,M+1))

  for i in range(M):

      x_1 = tri[i]
      x_2 = tri[i+1]
      h = x_2 -x_1
      grad_phi_1 = -1/h
      grad_phi_2 = 1/h
      local_stiff = np.zeros((2,2))

      for xi, w in zip(xi_q,w_q):

        xq = (x_2+x_1)/2 + h * (1/2) * xi

        local_stiff[0,0] += w * (grad_phi_1 * grad_phi_1 * a(xq)) * h * 1/2
        local_stiff[0,1] += w * (grad_phi_1 * grad_phi_2 * a(xq)) * h * 1/2
        local_stiff[1,0] += w * (grad_phi_2 * grad_phi_1 * a(xq)) * h * 1/2
        local_stiff[1,1] += w * (grad_phi_2 * grad_phi_2 * a(xq)) * h * 1/2

      stiff[i,i] += local_stiff[0,0]
      stiff[i,i+1] += local_stiff[0,1]
      stiff[i+1,i] += local_stiff[1,0]
      stiff[i+1,i+1] += local_stiff[1,1]

  stiff = stiff[1:-1,1:-1]

  return stiff


def a(x):
    if x < np.pi/2:
        return 1.0
    else:
        return 10000.0


In [ ]:
#@title Parareal functions
#parareal functions, CP/FP, computing propagators sequentially, Test/error functions to varify results, caluclating right hand side
#functions to prepare using CP/FP or parareal itself to save calculations, by calculating values that occur multiple times beforehand
#

#L2 error with quadrature

def L2_error(M, OMEGA, U, u, t, quad_order = 5):
  l_boundary, r_boundary = OMEGA

  U_full = np.zeros(M + 1)
  U_full[1:-1] = U

  xi_q, w_q = leggauss(quad_order)

  tri = mesh(M, OMEGA)

  Integral_phi_u = 0

  for i in range(M):
    x_1 = tri[i]
    x_2 = tri[i+1]
    h = x_2 -x_1

    for xi, w in zip(xi_q,w_q):

      xq = (x_2+x_1)/2 + h * (1/2) * xi

      phi_1 = (x_2-xq)/h
      phi_2 = (xq-x_1)/h

      Integral_phi_u += w * ((phi_1 * U_full[i] + phi_2 * U_full[i+1]) - u(xq,t))**2 * (1/2) * h

  return np.sqrt(Integral_phi_u)


def f_h(t):
  RHS = calculate_RHS(f, mesh, M, OMEGA, t)
  return solve(mass, RHS)

def prepare_CP(delta_T, A_h, P, R):
  B = delta_T * A_h
  P_i_sum = []
  R_T_A_h = R(B)
  for i in range(len(P)):
    P_i_sum.append(P[i](B))
  return P_i_sum, R_T_A_h


def matrix_from_scalar_function(phi, delta_T, eigenvalues, eigenvectors, eigenvectors_inverse):
  values = np.asarray(phi(delta_T * eigenvalues), dtype=np.float64)
  return (eigenvectors * values) @ eigenvectors_inverse


def prepare_R_spectral(delta_T, phi, eigenvalues, eigenvectors, eigenvectors_inverse):
  R_T_A_h = matrix_from_scalar_function(phi, delta_T, eigenvalues, eigenvectors, eigenvectors_inverse)
  return [], R_T_A_h


def prepare_rhs(C, f, mass, delta_t, T_intervall, M, OMEGA):

    RHS_values = {}

    t_0, t_end = T_intervall
    N_steps = int(round((t_end - t_0) / delta_t))
    T_n = np.linspace(t_0, t_end, N_steps + 1)
    for tn in T_n[:-1]:
        for c in C:
            t = round(float(tn + c * delta_t), 14)
            if t not in RHS_values:
                RHS = calculate_RHS(f,mesh,M,OMEGA,t)
                RHS_values[t] = solve(mass, RHS)
    return RHS_values


def CP(RHS_values, R_T_A_h, P_i_sum, T_n, delta_T, v, f_h, R, P, C, A_h):
  # Im einzigen Experiment gilt f(x,t) identisch 0. Dann ist kein RHS-Term nötig.
  if RHS_values is None:
    return R_T_A_h @ v

  B = delta_T * A_h
  rhs_sum = 0

  for i in range(len(P)):
    t = round(float(T_n + C[i] * delta_T),14)
    rhs_sum += (P_i_sum[i] @ RHS_values[t])


  return R_T_A_h @ v + delta_T * rhs_sum



def solve_on_grid(RHS_values, R_T_A_h, P_i_sum, CP, T_intervall, v_h, f_h, N, R, P, C, A_h):
  t_0, t_end = T_intervall
  delta_T = (t_end-t_0)/N
  T_n = np.linspace(t_0, t_end, N+1)
  grid_solutions = []
  grid_solutions.append(v_h)

  for i in range(N):
    grid_solutions.append(CP(RHS_values, R_T_A_h, P_i_sum, T_n[i], delta_T, grid_solutions[i], f_h, R, P, C, A_h))

  return grid_solutions



def calculate_U(RHS_values, R_T_A_h, P_i_sum, f, M, N, OMEGA, mesh, u_0, T, R, P, C, A_h, mass, v_h):

  l_boundary, r_boundary = OMEGA
  tri = mesh(M, OMEGA)
  inner_tri = tri[1:-1]
  t_0, t_end = T
  T_n = np.linspace(t_0, t_end, N+1)


  U = solve_on_grid(RHS_values, R_T_A_h, P_i_sum, CP, T, v_h, f_h, N, R, P, C, A_h)

  return U




In [ ]:
#@title Fine propagatos
# R,P_i rational functions and distinct real numbers C, for 2,3,4-stage LOBATTO IIIC schemes,
# 3-stage RDAU IIA, Backward Euler and SDIRK-22
#

def R_LobattoIIIC2(B):
    I = np.eye(B.shape[0])
    D = B @ B + 2*B + 2*I
    return 2*I @ solve(D, I)


def P1_LobattoIIIC2(B):
    I = np.eye(B.shape[0])
    D = B @ B + 2*B + 2*I
    return solve(D, I)


def P2_LobattoIIIC2(B):
    I = np.eye(B.shape[0])
    D = B @ B + 2*B + 2*I
    return (B + I) @ solve(D, I)


P_LobattoIIIC2 = [
    P1_LobattoIIIC2,
    P2_LobattoIIIC2
]

C_LobattoIIIC2 = [
    0.0,
    1.0
]



# ------------------------------------------------------------
# 3-stage Lobatto IIIC
# ------------------------------------------------------------

def R_LobattoIIIC3(B):
    I = np.eye(B.shape[0])

    D = (
        B @ B @ B
        + 6*(B @ B)
        + 18*B
        + 24*I
    )

    return (-6*B + 24*I) @ solve(D, I)


def P1_LobattoIIIC3(B):
    I = np.eye(B.shape[0])

    D = (
        B @ B @ B
        + 6*(B @ B)
        + 18*B
        + 24*I
    )

    return (4*I - B) @ solve(D, I)


def P2_LobattoIIIC3(B):
    I = np.eye(B.shape[0])

    D = (
        B @ B @ B
        + 6*(B @ B)
        + 18*B
        + 24*I
    )

    return (16*I + 4*B) @ solve(D, I)


def P3_LobattoIIIC3(B):
    I = np.eye(B.shape[0])

    D = (
        B @ B @ B
        + 6*(B @ B)
        + 18*B
        + 24*I
    )

    return (
        4*I
        + 3*B
        + B @ B
    ) @ solve(D, I)


P_LobattoIIIC3 = [
    P1_LobattoIIIC3,
    P2_LobattoIIIC3,
    P3_LobattoIIIC3
]

C_LobattoIIIC3 = [
    0.0,
    0.5,
    1.0
]



# ------------------------------------------------------------
# 4-stage Lobatto IIIC
# ------------------------------------------------------------

def R_LobattoIIIC4(B):
    I = np.eye(B.shape[0])

    B2 = B @ B
    B3 = B2 @ B
    B4 = B3 @ B

    D = (
        B4
        + 12*B3
        + 72*B2
        + 240*B
        + 360*I
    )

    return (
        12*B2
        - 120*B
        + 360*I
    ) @ solve(D, I)


def P1_LobattoIIIC4(B):
    I = np.eye(B.shape[0])

    B2 = B @ B
    B3 = B2 @ B
    B4 = B3 @ B

    D = (
        B4
        + 12*B3
        + 72*B2
        + 240*B
        + 360*I
    )

    return (
        B2
        - 10*B
        + 30*I
    ) @ solve(D, I)


def P2_LobattoIIIC4(B):
    I = np.eye(B.shape[0])
    sqrt5 = np.sqrt(5)

    B2 = B @ B
    B3 = B2 @ B
    B4 = B3 @ B

    D = (
        B4
        + 12*B3
        + 72*B2
        + 240*B
        + 360*I
    )

    numerator = (
        (5/2)*(1-sqrt5)*B2
        + (25 - 15*sqrt5)*B
        + 150*I
    )

    return numerator @ solve(D, I)


def P3_LobattoIIIC4(B):
    I = np.eye(B.shape[0])
    sqrt5 = np.sqrt(5)

    B2 = B @ B
    B3 = B2 @ B
    B4 = B3 @ B

    D = (
        B4
        + 12*B3
        + 72*B2
        + 240*B
        + 360*I
    )

    numerator = (
        (5/2)*(1+sqrt5)*B2
        + (25 + 15*sqrt5)*B
        + 150*I
    )

    return numerator @ solve(D, I)


def P4_LobattoIIIC4(B):
    I = np.eye(B.shape[0])

    B2 = B @ B
    B3 = B2 @ B
    B4 = B3 @ B

    D = (
        B4
        + 12*B3
        + 72*B2
        + 240*B
        + 360*I
    )

    return (
        B3
        + 6*B2
        + 20*B
        + 30*I
    ) @ solve(D, I)


P_LobattoIIIC4 = [
    P1_LobattoIIIC4,
    P2_LobattoIIIC4,
    P3_LobattoIIIC4,
    P4_LobattoIIIC4
]

C_LobattoIIIC4 = [
    0.0,
    (5-np.sqrt(5))/10,
    (5+np.sqrt(5))/10,
    1.0
]



# ------------------------------------------------------------
# 3-stage Radau IIA
# ------------------------------------------------------------

def R_RadauIIA3(B):
    I = np.eye(B.shape[0])

    B2 = B @ B
    B3 = B2 @ B

    D = (
        B3
        + 9*B2
        + 36*B
        + 60*I
    )

    return (
        3*B2
        - 24*B
        + 60*I
    ) @ solve(D, I)


def P1_RadauIIA3(B):
    I = np.eye(B.shape[0])
    sqrt6 = np.sqrt(6)

    B2 = B @ B
    B3 = B2 @ B

    D = (
        B3
        + 9*B2
        + 36*B
        + 60*I
    )

    numerator = (
        (1 - 8*sqrt6/3)*B
        + (80/3 - 5*sqrt6/3)*I
    )

    return numerator @ solve(D, I)


def P2_RadauIIA3(B):
    I = np.eye(B.shape[0])
    sqrt6 = np.sqrt(6)

    B2 = B @ B
    B3 = B2 @ B

    D = (
        B3
        + 9*B2
        + 36*B
        + 60*I
    )

    numerator = (
        (1 + 8*sqrt6/3)*B
        + (80/3 + 5*sqrt6/3)*I
    )

    return numerator @ solve(D, I)


def P3_RadauIIA3(B):
    I = np.eye(B.shape[0])

    B2 = B @ B
    B3 = B2 @ B

    D = (
        B3
        + 9*B2
        + 36*B
        + 60*I
    )

    return (
        B2
        + 4*B
        + (20/3)*I
    ) @ solve(D, I)


P_RadauIIA3 = [
    P1_RadauIIA3,
    P2_RadauIIA3,
    P3_RadauIIA3
]

C_RadauIIA3 = [
    (4-np.sqrt(6))/10,
    (4+np.sqrt(6))/10,
    1.0
]



# ------------------------------------------------------------
# Backward Euler
# ------------------------------------------------------------

def R_BE(B):
    I = np.eye(B.shape[0])
    return solve(I + B, I)


P_BE = [R_BE]
C_BE = [1.0]

# ------------------------------------------------------------
# SDIRK-22
# ------------------------------------------------------------

gamma_SDIRK22 = (2.0 - np.sqrt(2.0)) / 2.0


def R_SDIRK22(B):
    I = np.eye(B.shape[0])
    Q = I + gamma_SDIRK22 * B

    return (I + (2.0 * gamma_SDIRK22 - 1.0) * B) @ solve(Q @ Q, I)


def P1_SDIRK22(B):
    I = np.eye(B.shape[0])
    Q = I + gamma_SDIRK22 * B

    return (1.0 - gamma_SDIRK22) * solve(Q @ Q, I)


def P2_SDIRK22(B):
    I = np.eye(B.shape[0])
    Q = I + gamma_SDIRK22 * B

    return gamma_SDIRK22 * solve(Q, I)


P_SDIRK22 = [
    P1_SDIRK22,
    P2_SDIRK22
]

C_SDIRK22 = [
    gamma_SDIRK22,
    1.0
]


# ------------------------------------------------------------
# Skalare Auswertung der Stabilitätsfunktion des
# 4-stage Lobatto IIIC Verfahrens für die Spektraldarstellung
# ------------------------------------------------------------

def R_LobattoIIIC4_scalar(s):
    s = np.asarray(s, dtype=np.float64)
    denominator = (((s + 12.0) * s + 72.0) * s + 240.0) * s + 360.0
    numerator = (12.0 * s - 120.0) * s + 360.0
    return numerator / denominator

# ------------------------------------------------------------
# Skalare Auswertung der Stabilitätsfunktion des
# 3-stage Radau IIA Verfahrens für die Spektraldarstellung
# ------------------------------------------------------------

def R_RadauIIA3_scalar(s):
    s = np.asarray(s, dtype=np.float64)

    numerator = (3.0 * s - 24.0) * s + 60.0
    denominator = ((s + 9.0) * s + 36.0) * s + 60.0

    return numerator / denominator


In [ ]:
#@title FP call
# Ditionary to easily call different FP´s and CP´s when using the parareal-algorithm

FP = {
    "LobattoIIIC2": {
        "R": R_LobattoIIIC2,
        "P": P_LobattoIIIC2,
        "C": C_LobattoIIIC2,
        "title": "two-stage Lobatto IIIC"
    },

    "LobattoIIIC3": {
        "R": R_LobattoIIIC3,
        "P": P_LobattoIIIC3,
        "C": C_LobattoIIIC3,
        "title": "three-stage Lobatto IIIC"
    },

    "LobattoIIIC4": {
        "R": R_LobattoIIIC4,
        "P": P_LobattoIIIC4,
        "C": C_LobattoIIIC4,
        "R_scalar": R_LobattoIIIC4_scalar,
        "title": "four-stage Lobatto IIIC"
    },

    "RadauIIA3": {
        "R": R_RadauIIA3,
        "P": P_RadauIIA3,
        "C": C_RadauIIA3,
        "R_scalar": R_RadauIIA3_scalar,
        "title": "three-stage Radau IIA"
    },

    "BackwardEuler": {
        "R": R_BE,
        "P": P_BE,
        "C": C_BE,
        "title": "Backward Euler"
    },

    "SDIRK22": {
        "R": R_SDIRK22,
        "P": P_SDIRK22,
        "C": C_SDIRK22,
        "title": "SDIRK-22"
    }
}

SDIRK22 = FP["SDIRK22"]
BE = FP["BackwardEuler"]

In [ ]:
#@title parareal error function

#L2-error for two FEM-functions defined with mass matrix and a parareal function, which saves the parareal to fine solution error

def L2_for_FEM(U1, U2, mass):
    errors = []
    for u1, u2 in zip(U1, U2):
        e = u1 - u2
        errors.append(np.sqrt(e.T @ mass @ e))
    return max(errors)


def parareal_fehlervergleich(RHS_values_coarse, RHS_values_fine, R_T_A_h_coarse, R_T_A_h_fine, P_i_sum_coarse, P_i_sum_fnie, CP, T_intervall, U_fine, M, N, J, u_0, f, R_C, R_F, P_C, P_F, C_C, C_F, K, A_h, stiff, mass, v_h):

  t_0, t_end =T_intervall
  T_n_coarse = np.linspace(t_0, t_end, int(N/J)+1)
  delta_T = (t_end - t_0)/(int(N/J))

  error_hist = []

  U = solve_on_grid(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, CP, T_intervall, v_h, f_h, int(N/J), R_C, P_C, C_C, A_h)
  for k in range(K):

    U_alt = U.copy()
    Fine_solutions = [v_h]

    for i in range(int(N/J)):
        U_n_j = U[i]
        U_n1_k = solve_on_grid(RHS_values_fine, R_T_A_h_fine, P_i_sum_fnie, CP, (T_n_coarse[i],T_n_coarse[i+1]), U_n_j, f_h, J, R_F, P_F, C_F, A_h)[-1]
        Fine_solutions.append(U_n1_k)

    for i in range(int(N/J)):
      U[i+1] = CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U[i], f_h, R_C, P_C, C_C, A_h) + Fine_solutions[i+1] - CP(RHS_values_coarse, R_T_A_h_coarse, P_i_sum_coarse, T_n_coarse[i], delta_T, U_alt[i], f_h, R_C, P_C, C_C, A_h)

    err = L2_for_FEM(U,U_fine, mass)
    error_hist.append(err)

  return U, error_hist

In [ ]:
#@title OCP stability function

#importing the parameters for the OCP stability function, building the stability funtion, and building the functions P_i(s).
#At the moment only for one function P_i, since q=1 for every optimization we have done in the bachelorthesis.


def matrix_polynomial(B, coeffs):
    I = np.eye(B.shape[0])
    result = coeffs[0] * I
    B_power = I

    for coeff in coeffs[1:]:
        B_power = B_power @ B
        result = result + coeff * B_power

    return result


#The input for this is the filename of the file that was produced by the OCP-algorithm,
#containing the information about the stability-fucntion.
#This is done because otherwise you would either have to build the OCP algorithm inside the parareal function, or
#if you printed out the parameters there would be round off errors.

def load_OCP(filename):

    saved = torch.load(filename, map_location="cpu", weights_only=False)

    if isinstance(saved, list):
        saved = saved[0]

    a = saved["a"].detach().cpu().numpy().astype(np.float64)
    b = saved["b"].detach().cpu().numpy().astype(np.float64)

    def R_scalar(s):
        s = np.asarray(s, dtype=np.float64)
        numerator = np.polynomial.polynomial.polyval(s, a)
        denominator = np.polynomial.polynomial.polyval(s, b)
        return numerator / denominator

    def R(B):
        I = np.eye(B.shape[0])

        numerator   = matrix_polynomial(B, a)
        denominator = matrix_polynomial(B, b)

        return numerator @ solve(denominator, I)

    # Für deine jetzigen q=1-OCPs
    def P1(B):
        I = np.eye(B.shape[0])
        return solve(B, I - R(B))

    return {
        "R": R,
        "P": [P1],
        "C": [1],
        "R_scalar": R_scalar,
        "a": a,
        "b": b
    }


